# 03 - Batch and Arrow Handoff Fast Path

This notebook demonstrates the Arrow batch portfolio API. It compares the row API with the Arrow batch dispatcher and inspects diagnostics that show accepted input-row dataclasses were not materialized on the handoff path.


In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd()
PACKAGE_ROOT = None
REPO_ROOT = None
for candidate in (HERE, *HERE.parents):
    if (candidate / "src" / "frtb_sbm").exists():
        PACKAGE_ROOT = candidate
        REPO_ROOT = candidate.parent.parent if candidate.parent.name == "packages" else candidate
        break
    if (candidate / "packages" / "frtb-sbm" / "src" / "frtb_sbm").exists():
        REPO_ROOT = candidate
        PACKAGE_ROOT = candidate / "packages" / "frtb-sbm"
        break
if PACKAGE_ROOT is None:
    raise RuntimeError("Could not locate frtb-sbm package root")
workspace_src_paths = tuple(sorted((REPO_ROOT / "packages").glob("*/src"))) if REPO_ROOT else ()
for path in (
    PACKAGE_ROOT / "examples",
    PACKAGE_ROOT,
    PACKAGE_ROOT / "src",
    *workspace_src_paths,
    REPO_ROOT,
):
    if path is not None and str(path) not in sys.path:
        sys.path.insert(0, str(path))
PACKAGE_ROOT

In [ ]:
try:
    from IPython.display import Markdown, display
except ImportError:

    class Markdown(str):
        pass

    def display(value: object) -> None:
        print(value)


from frtb_sbm import calculate_sbm_capital, serialize_sbm_result
from frtb_sbm.arrow_batch import calculate_sbm_portfolio_capital_from_arrow_tables
from sbm_notebook_data import (
    arrow_tables_for_sensitivities,
    markdown_table,
    notebook_context,
    portfolio_sample_sensitivities,
)

In [ ]:
context = notebook_context("sbm-arrow-table-notebook")
sensitivities = portfolio_sample_sensitivities()
arrow_tables = arrow_tables_for_sensitivities(sensitivities)

row_result = calculate_sbm_capital(sensitivities, context=context)
arrow_calculation = calculate_sbm_portfolio_capital_from_arrow_tables(arrow_tables, context=context)
row_payload = serialize_sbm_result(row_result)
arrow_payload = serialize_sbm_result(arrow_calculation.result)

display(
    Markdown(
        markdown_table(
            ("Metric", "Row API", "Arrow table API"),
            (
                ("Total capital", row_payload["total_capital"], arrow_payload["total_capital"]),
                (
                    "Input hash",
                    row_payload["input_hash"][:16] + "...",
                    arrow_payload["input_hash"][:16] + "...",
                ),
                (
                    "Risk-class records",
                    len(row_payload["risk_classes"]),
                    len(arrow_payload["risk_classes"]),
                ),
                (
                    "Accepted row dataclasses materialized",
                    "n/a",
                    arrow_calculation.accepted_row_dataclasses_materialized,
                ),
                ("Arrow table count", "n/a", len(arrow_tables)),
            ),
        )
    )
)

In [ ]:
display(
    Markdown(
        markdown_table(
            ("Invariant", "Result"),
            (
                (
                    "Total capital matches",
                    row_payload["total_capital"] == arrow_payload["total_capital"],
                ),
                ("Input hash matches", row_payload["input_hash"] == arrow_payload["input_hash"]),
                (
                    "No accepted row dataclasses materialized",
                    arrow_calculation.accepted_row_dataclasses_materialized == 0,
                ),
            ),
        )
    )
)

In [ ]:
diagnostic_rows = (
    (
        diagnostic.risk_class.value,
        diagnostic.risk_measure.value,
        diagnostic.input_count,
        diagnostic.batch_count,
        diagnostic.accepted_row_dataclasses_materialized,
    )
    for diagnostic in arrow_calculation.path_diagnostics
)
display(
    Markdown(
        markdown_table(
            ("Risk class", "Measure", "Input count", "Batch count", "Rows materialized"),
            diagnostic_rows,
        )
    )
)

In [ ]:
sample_schema = arrow_tables[0].accepted.schema
display(
    Markdown(
        markdown_table(
            ("Column", "Arrow type"),
            ((field.name, str(field.type)) for field in sample_schema),
        )
    )
)